<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# Biểu đồ Waffle (Waffle Charts), Word Cloud và Biểu đồ hồi quy (Regression Plots)


Thời gian ước tính cần thiết: **30** phút
    

## Mục tiêu

Sau khi hoàn thành bài thực hành này, bạn sẽ có thể:

* Tạo Word cloud và biểu đồ Waffle (Waffle charts)
* Tạo biểu đồ hồi quy (regression plots) bằng thư viện Seaborn

## Mục lục

<div class="alert alert-block alert-info" style="margin-top: 20px">

1. [Khám phá bộ dữ liệu với *p*andas](#0)<br>
2. [Tải xuống và chuẩn bị dữ liệu](#2)<br>
3. [Trực quan hóa dữ liệu bằng Matplotlib](#4) <br>
4. [Biểu đồ Waffle (Waffle Charts)](#6) <br>
5. [Word Cloud](#8) <br>
7. [Biểu đồ hồi quy (Regression Plots)](#10) <br> 
</div>

# Khám phá bộ dữ liệu với *pandas* và Matplotlib<a id="0"></a>

Bộ công cụ: Khóa học phụ thuộc nhiều vào [*pandas*](http://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01) và [*Numpy*](http://www.numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01) để xử lý (data wrangling), phân tích và trực quan hóa dữ liệu. Thư viện vẽ biểu đồ chính mà chúng ta sẽ tìm hiểu trong khóa học là [Matplotlib](http://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01).

Bộ dữ liệu: Nhập cư vào Canada từ năm 1980 đến 2013 - [International migration flows to and from selected countries - The 2015 revision](http://www.un.org/en/development/desa/population/migration/data/empirical2/migrationflows.shtml?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01) từ trang web của Liên Hợp Quốc

Bộ dữ liệu chứa dữ liệu hằng năm về các luồng người di cư quốc tế do các quốc gia đến ghi nhận. Dữ liệu thể hiện cả luồng nhập và luồng xuất theo nơi sinh, quốc tịch hoặc nơi cư trú trước đó / tiếp theo, đối với cả người nước ngoài và công dân. Trong bài thực hành này, chúng ta sẽ tập trung vào dữ liệu nhập cư vào Canada.

# Tải xuống và chuẩn bị dữ liệu <a id="2"></a>

Việc đầu tiên chúng ta làm là cài đặt **openpyxl** (trước đây là **xlrd**), một module mà *pandas* cần để đọc các tệp Excel.

In [ ]:
!mamba install openpyxl==3.0.9 -y

Import các module chính:

In [ ]:
import numpy as np  # useful for many scientific computing in Python
import pandas as pd # primary data structure library
from PIL import Image # converting images into arrays

Tải xuống bộ dữ liệu nhập cư Canada và đọc nó vào một dataframe của *pandas*.

In [ ]:
df_can = pd.read_excel(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.xlsx',
    sheet_name='Canada by Citizenship',
    skiprows=range(20),
    skipfooter=2)

print('Data downloaded and read into a dataframe!')

Hãy xem qua năm phần tử đầu tiên trong bộ dữ liệu của chúng ta

In [ ]:
df_can.head()

Hãy tìm xem có bao nhiêu mục (entry) trong bộ dữ liệu của chúng ta

In [ ]:
# print the dimensions of the dataframe
print(df_can.shape)

Làm sạch dữ liệu. Chúng ta sẽ thực hiện một số thay đổi đối với bộ dữ liệu gốc để việc tạo các hình ảnh trực quan dễ dàng hơn. Hãy tham khảo *Introduction to Matplotlib and Line Plots* và *Area Plots, Histograms, and Bar Plots* để biết mô tả chi tiết về bước tiền xử lý này.

In [ ]:
# clean up the dataset to remove unnecessary columns (eg. REG) 
df_can.drop(['AREA','REG','DEV','Type','Coverage'], axis = 1, inplace = True)

# let's rename the columns so that they make sense
df_can.rename (columns = {'OdName':'Country', 'AreaName':'Continent','RegName':'Region'}, inplace = True)

# for sake of consistency, let's also make all column labels of type string
df_can.columns = list(map(str, df_can.columns))

# set the country name as index - useful for quickly looking up countries using .loc method
df_can.set_index('Country', inplace = True)

# add total column
df_can['Total'] =  df_can.sum (axis = 1)

# years that we will be using in this lesson - useful for plotting later on
years = list(map(str, range(1980, 2014)))
print ('data dimensions:', df_can.shape)

# Trực quan hóa dữ liệu bằng Matplotlib<a id="4"></a>

Import và thiết lập `matplotlib`:

In [ ]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches # needed for waffle Charts

mpl.style.use('ggplot') # optional: for ggplot-like style

# check for latest version of Matplotlib
print ('Matplotlib version: ', mpl.__version__) # >= 2.0.0

# Biểu đồ Waffle (Waffle Charts) <a id="6"></a>


`Waffle chart` là một dạng trực quan hóa thú vị thường được tạo để hiển thị tiến độ hướng tới các mục tiêu. Nó thường là một lựa chọn hiệu quả khi bạn muốn thêm các tính năng trực quan thú vị vào một hình ảnh chủ yếu gồm các ô (cell), chẳng hạn như một dashboard Excel.

Hãy xem lại nghiên cứu tình huống trước đó về Đan Mạch (Denmark), Na Uy (Norway) và Thụy Điển (Sweden).

In [ ]:
# let's create a new dataframe for these three countries 
df_dsn = df_can.loc[['Denmark', 'Norway', 'Sweden'], :]

# let's take a look at our dataframe
df_dsn

Đáng tiếc là, không giống R, các biểu đồ `waffle` không được tích hợp sẵn trong bất kỳ thư viện trực quan hóa nào của Python. Vì vậy, chúng ta sẽ học cách tạo chúng từ đầu.

**Bước 1.** Bước đầu tiên để tạo một waffle chart là xác định tỷ lệ của mỗi danh mục so với tổng.

In [ ]:
# compute the proportion of each category with respect to the total
total_values = df_dsn['Total'].sum()
category_proportions = df_dsn['Total'] / total_values

# print out proportions
pd.DataFrame({"Category Proportion": category_proportions})

**Bước 2.** Bước thứ hai là xác định kích thước tổng thể của biểu đồ `waffle`.

In [ ]:
width = 40 # width of chart
height = 10 # height of chart

total_num_tiles = width * height # total number of tiles

print(f'Total number of tiles is {total_num_tiles}.')

**Bước 3.** Bước thứ ba là dùng tỷ lệ của mỗi danh mục để xác định số ô (tile) tương ứng của nó.

In [ ]:
# compute the number of tiles for each category
tiles_per_category = (category_proportions * total_num_tiles).round().astype(int)

# print out number of tiles per category
pd.DataFrame({"Number of tiles": tiles_per_category})

Dựa trên các tỷ lệ đã tính, Denmark sẽ chiếm 129 ô của biểu đồ `waffle`, Norway sẽ chiếm 77 ô và Sweden sẽ chiếm 194 ô.

**Bước 4.** Bước thứ tư là tạo một ma trận (matrix) giống với biểu đồ `waffle` và điền dữ liệu vào đó.

In [ ]:
# initialize the waffle chart as an empty matrix
waffle_chart = np.zeros((height, width), dtype = np.uint)

# define indices to loop through waffle chart
category_index = 0
tile_index = 0

# populate the waffle chart
for col in range(width):
    for row in range(height):
        tile_index += 1

        # if the number of tiles populated for the current category is equal to its corresponding allocated tiles...
        if tile_index > sum(tiles_per_category[0:category_index]):
            # ...proceed to the next category
            category_index += 1       
            
        # set the class value to an integer, which increases with class
        waffle_chart[row, col] = category_index
        
print ('Waffle chart populated!')

Hãy xem qua ma trận trông như thế nào.

In [ ]:
waffle_chart

Đúng như mong đợi, ma trận gồm ba danh mục và tổng số phần tử của mỗi danh mục khớp với tổng số ô được phân bổ cho danh mục đó.

**Bước 5.** Ánh xạ ma trận của biểu đồ `waffle` thành hình ảnh trực quan.

In [ ]:
# instantiate a new figure object
fig = plt.figure()

# use matshow to display the waffle chart
colormap = plt.cm.coolwarm
plt.matshow(waffle_chart, cmap=colormap)
plt.colorbar()
plt.show()

**Bước 6.** Làm đẹp biểu đồ.

In [ ]:
# instantiate a new figure object
fig = plt.figure()

# use matshow to display the waffle chart
colormap = plt.cm.coolwarm
plt.matshow(waffle_chart, cmap=colormap)
plt.colorbar()

# get the axis
ax = plt.gca()

# set minor ticks
ax.set_xticks(np.arange(-.5, (width), 1), minor=True)
ax.set_yticks(np.arange(-.5, (height), 1), minor=True)
    
# add gridlines based on minor ticks
ax.grid(which='minor', color='w', linestyle='-', linewidth=2)

plt.xticks([])
plt.yticks([])
plt.show()

**Bước 7.** Tạo chú giải (legend) và thêm nó vào biểu đồ.

In [ ]:
# instantiate a new figure object
fig = plt.figure()

# use matshow to display the waffle chart
colormap = plt.cm.coolwarm
plt.matshow(waffle_chart, cmap=colormap)
plt.colorbar()

# get the axis
ax = plt.gca()

# set minor ticks
ax.set_xticks(np.arange(-.5, (width), 1), minor=True)
ax.set_yticks(np.arange(-.5, (height), 1), minor=True)
    
# add gridlines based on minor ticks
ax.grid(which='minor', color='w', linestyle='-', linewidth=2)

plt.xticks([])
plt.yticks([])

# compute cumulative sum of individual categories to match color schemes between chart and legend
values_cumsum = np.cumsum(df_dsn['Total'])
total_values = values_cumsum[len(values_cumsum) - 1]

# create legend
legend_handles = []
for i, category in enumerate(df_dsn.index.values):
    label_str = category + ' (' + str(df_dsn['Total'][i]) + ')'
    color_val = colormap(float(values_cumsum[i])/total_values)
    legend_handles.append(mpatches.Patch(color=color_val, label=label_str))

# add legend to chart
plt.legend(handles=legend_handles,
           loc='lower center', 
           ncol=len(df_dsn.index.values),
           bbox_to_anchor=(0., -0.2, 0.95, .1)
          )
plt.show()

Và đây rồi! Một biểu đồ `waffle` trông thật đẹp mắt và *ngon lành*, bạn thấy không?

Sẽ rất kém hiệu quả nếu phải lặp lại bảy bước này mỗi khi chúng ta muốn tạo một biểu đồ `waffle`. Vì vậy, hãy gộp cả bảy bước thành một hàm có tên *create_waffle_chart*. Hàm này sẽ nhận các tham số đầu vào sau:

> 1. **categories**: Các danh mục hoặc lớp duy nhất trong dataframe.
> 2. **values**: Các giá trị tương ứng với các danh mục hoặc lớp.
> 3. **height**: Chiều cao được định nghĩa của waffle chart.
> 4. **width**: Chiều rộng được định nghĩa của waffle chart.
> 5. **colormap**: Lớp Colormap
> 6. **value_sign**: Để làm cho hàm của chúng ta có tính tổng quát hơn, chúng ta sẽ thêm tham số này để xử lý các ký hiệu có thể đi kèm với một giá trị như %, $, v.v. **value_sign** có giá trị mặc định là chuỗi rỗng.

In [ ]:
def create_waffle_chart(categories, values, height, width, colormap, value_sign=''):

    # compute the proportion of each category with respect to the total
    total_values = sum(values)
    category_proportions = [(float(value) / total_values) for value in values]

    # compute the total number of tiles
    total_num_tiles = width * height # total number of tiles
    print ('Total number of tiles is', total_num_tiles)
    
    # compute the number of tiles for each catagory
    tiles_per_category = [round(proportion * total_num_tiles) for proportion in category_proportions]

    # print out number of tiles per category
    for i, tiles in enumerate(tiles_per_category):
        print (df_dsn.index.values[i] + ': ' + str(tiles))
    
    # initialize the waffle chart as an empty matrix
    waffle_chart = np.zeros((height, width))

    # define indices to loop through waffle chart
    category_index = 0
    tile_index = 0

    # populate the waffle chart
    for col in range(width):
        for row in range(height):
            tile_index += 1

            # if the number of tiles populated for the current category 
            # is equal to its corresponding allocated tiles...
            if tile_index > sum(tiles_per_category[0:category_index]):
                # ...proceed to the next category
                category_index += 1       
            
            # set the class value to an integer, which increases with class
            waffle_chart[row, col] = category_index
    
    # instantiate a new figure object
    fig = plt.figure()

    # use matshow to display the waffle chart
    colormap = plt.cm.coolwarm
    plt.matshow(waffle_chart, cmap=colormap)
    plt.colorbar()

    # get the axis
    ax = plt.gca()

    # set minor ticks
    ax.set_xticks(np.arange(-.5, (width), 1), minor=True)
    ax.set_yticks(np.arange(-.5, (height), 1), minor=True)
    
    # add dridlines based on minor ticks
    ax.grid(which='minor', color='w', linestyle='-', linewidth=2)

    plt.xticks([])
    plt.yticks([])

    # compute cumulative sum of individual categories to match color schemes between chart and legend
    values_cumsum = np.cumsum(values)
    total_values = values_cumsum[len(values_cumsum) - 1]

    # create legend
    legend_handles = []
    for i, category in enumerate(categories):
        if value_sign == '%':
            label_str = category + ' (' + str(values[i]) + value_sign + ')'
        else:
            label_str = category + ' (' + value_sign + str(values[i]) + ')'
            
        color_val = colormap(float(values_cumsum[i])/total_values)
        legend_handles.append(mpatches.Patch(color=color_val, label=label_str))

    # add legend to chart
    plt.legend(
        handles=legend_handles,
        loc='lower center', 
        ncol=len(categories),
        bbox_to_anchor=(0., -0.2, 0.95, .1)
    )
    plt.show()

Bây giờ để tạo một biểu đồ `waffle`, tất cả những gì chúng ta cần làm là gọi hàm `create_waffle_chart`. Hãy định nghĩa các tham số đầu vào:

In [ ]:
width = 40 # width of chart
height = 10 # height of chart

categories = df_dsn.index.values # categories
values = df_dsn['Total'] # correponding values of categories

colormap = plt.cm.coolwarm # color map class

Và bây giờ hãy gọi hàm của chúng ta để tạo một biểu đồ `waffle`.

In [ ]:
create_waffle_chart(categories, values, height, width, colormap)

Dường như có một gói Python mới để tạo `waffle charts` có tên [PyWaffle](https://github.com/ligyxy/PyWaffle), nhưng có vẻ như repository này vẫn đang được xây dựng. Tuy nhiên, bạn cứ thoải mái xem thử và trải nghiệm nó.

# Word Cloud <a id="8"></a>


Word cloud (còn được gọi là text cloud hoặc tag cloud) hoạt động theo cách đơn giản: một từ cụ thể xuất hiện càng nhiều trong một nguồn dữ liệu văn bản (chẳng hạn như một bài phát biểu, bài đăng blog hoặc cơ sở dữ liệu), thì nó càng xuất hiện lớn và đậm hơn trong word cloud.

May mắn là đã có sẵn một gói Python để tạo `word` cloud. Gói này có tên `word_cloud`, được phát triển bởi **Andreas Mueller**. Bạn có thể tìm hiểu thêm về gói này qua [liên kết](https://github.com/amueller/word_cloud/) này.

Hãy dùng gói này để học cách tạo word cloud cho một tài liệu văn bản nhất định.

Đầu tiên, hãy cài đặt gói này.

In [ ]:
# install wordcloud
!pip3 install wordcloud==1.8.1

# import package and its set of stopwords
from wordcloud import WordCloud, STOPWORDS

print ('Wordcloud is installed and imported!')

`Word` cloud thường được dùng để phân tích và trực quan hóa dữ liệu văn bản ở mức tổng quan. Do đó, hãy tạm rời khỏi bộ dữ liệu nhập cư và làm việc với một ví dụ liên quan đến phân tích dữ liệu văn bản. Hãy thử phân tích một tiểu thuyết ngắn của **Lewis Carroll** có tên *Alice's Adventures in Wonderland* (Alice ở xứ sở thần tiên). Hãy tiếp tục và tải xuống một tệp _.txt_ của tiểu thuyết này.

In [ ]:
import urllib

# open the file and read it into a variable alice_novel
alice_novel = urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/alice_novel.txt').read().decode("utf-8")

Tiếp theo, hãy sử dụng các stopword mà chúng ta đã import từ `word_cloud`. Chúng ta dùng hàm *set* để loại bỏ mọi stopword dư thừa.

In [ ]:
stopwords = set(STOPWORDS)

Tạo một đối tượng word cloud và sinh ra một word cloud. Để đơn giản, chúng ta sẽ tạo word cloud chỉ bằng 2000 từ đầu tiên của tiểu thuyết.

In [ ]:
# instantiate a word cloud object
alice_wc = WordCloud(
    background_color='white',
    max_words=2000,
    stopwords=stopwords
)

# generate the word cloud
alice_wc.generate(alice_novel)

Tuyệt vời! Giờ đây `word` cloud đã được tạo, hãy trực quan hóa nó.

In [ ]:
# display the word cloud
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

Thú vị! Như vậy trong 2000 từ đầu tiên của tiểu thuyết, các từ phổ biến nhất là **Alice**, **said**, **little**, **Queen**, và cứ tiếp tục như vậy. Hãy thay đổi kích thước của cloud để chúng ta có thể thấy rõ hơn một chút các từ ít xuất hiện hơn.

In [ ]:
fig = plt.figure(figsize=(14, 18))

# display the cloud
plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

Tốt hơn nhiều! Tuy nhiên, **said** thực sự không phải là một từ mang nhiều thông tin. Vì vậy, hãy thêm nó vào các stopword của chúng ta và tạo lại cloud.

In [ ]:
stopwords.add('said') # add the words said to stopwords

# re-generate the word cloud
alice_wc.generate(alice_novel)

# display the cloud
fig = plt.figure(figsize=(14, 18))

plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

Xuất sắc! Trông thật thú vị! Một điều thú vị khác mà bạn có thể thực hiện với gói `word_cloud` là chồng các từ lên một mặt nạ (mask) có hình dạng bất kỳ. Hãy dùng một mask hình Alice và chú thỏ của cô. Chúng tôi đã tạo sẵn mask cho bạn, vì vậy hãy tải xuống và đặt tên là *alice_mask.png*.

In [ ]:
# save mask to alice_mask
alice_mask = np.array(Image.open(urllib.request.urlopen('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%204/images/alice_mask.png')))

Hãy xem mask trông như thế nào.

In [ ]:
fig = plt.figure(figsize=(14, 18))

plt.imshow(alice_mask, cmap=plt.cm.gray, interpolation='bilinear')
plt.axis('off')
plt.show()

Việc tạo hình cho `word` cloud theo mask rất đơn giản khi dùng gói `word_cloud`. Để đơn giản, chúng ta sẽ tiếp tục dùng 2000 từ đầu tiên của tiểu thuyết.

In [ ]:
# instantiate a word cloud object
alice_wc = WordCloud(background_color='white', max_words=2000, mask=alice_mask, stopwords=stopwords)

# generate the word cloud
alice_wc.generate(alice_novel)

# display the word cloud
fig = plt.figure(figsize=(14, 18))

plt.imshow(alice_wc, interpolation='bilinear')
plt.axis('off')
plt.show()

Thật ấn tượng!

Đáng tiếc là bộ dữ liệu nhập cư của chúng ta không có dữ liệu văn bản nào, nhưng có chí thì nên. Hãy tạo dữ liệu văn bản mẫu từ bộ dữ liệu nhập cư, giả sử là dữ liệu văn bản gồm 90 từ.

Hãy nhớ lại dữ liệu của chúng ta trông như thế nào.

In [ ]:
df_can.head()

Và tổng lượng nhập cư từ 1980 đến 2013 là bao nhiêu?

In [ ]:
total_immigration = df_can['Total'].sum()
total_immigration

Với các quốc gia có tên chỉ gồm một từ, hãy nhân bản (duplicate) tên của mỗi quốc gia dựa trên mức độ đóng góp của quốc gia đó vào tổng lượng nhập cư.

In [ ]:
max_words = 90
word_string = ''
for country in df_can.index.values:
    # check if country's name is a single-word name
    if country.count(" ") == 0:
        repeat_num_times = int(df_can.loc[country, 'Total'] / total_immigration * max_words)
        word_string = word_string + ((country + ' ') * repeat_num_times)

# display the generated text
word_string

Chúng ta không xử lý bất kỳ stopword nào ở đây, vì vậy không cần truyền chúng khi tạo word cloud.

In [ ]:
# create the word cloud
wordcloud = WordCloud(background_color='white').generate(word_string)

print('Word cloud created!')

In [ ]:
# display the cloud
plt.figure(figsize=(14, 18))

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

Theo word cloud ở trên, có vẻ như phần lớn những người nhập cư đến từ một trong 15 quốc gia được hiển thị trong word cloud. Một hình ảnh trực quan thú vị mà bạn có thể xây dựng là có thể dùng bản đồ Canada làm mask và chồng word cloud lên trên bản đồ Canada. Đó sẽ là một hình ảnh trực quan thú vị để xây dựng!

# Biểu đồ hồi quy (Regression Plots) <a id="10"></a>


> Seaborn là một thư viện trực quan hóa của Python dựa trên matplotlib. Nó cung cấp một giao diện cấp cao để vẽ các đồ thị thống kê đẹp mắt. Bạn có thể tìm hiểu thêm về *seaborn* qua [liên kết](https://seaborn.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01) này và tìm hiểu thêm về các regression plot của *seaborn* qua [liên kết](http://seaborn.pydata.org/generated/seaborn.regplot.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01) này.

Trong bài thực hành *Pie Charts, Box Plots, Scatter Plots, and Bubble Plots*, chúng ta đã học cách tạo scatter plot rồi khớp một đường hồi quy (regression line). Cần khoảng ~20 dòng code để tạo scatter plot cùng với đường hồi quy khớp. Trong phần cuối cùng này, chúng ta sẽ khám phá *seaborn* và xem việc tạo các đường hồi quy và đường khớp bằng thư viện này hiệu quả đến mức nào!

Đầu tiên, hãy cài đặt *seaborn*

In [ ]:
# install seaborn
# !pip3 install seaborn

# import library
import seaborn as sns

print('Seaborn installed and imported!')

Tạo một dataframe mới lưu tổng số người nhập cư hợp pháp (landed immigrants) vào Canada mỗi năm từ 1980 đến 2013.

In [ ]:
# we can use the sum() method to get the total population per year
df_tot = pd.DataFrame(df_can[years].sum(axis=0))

# change the years to type float (useful for regression later on)
df_tot.index = map(float, df_tot.index)

# reset the index to put in back in as a column in the df_tot dataframe
df_tot.reset_index(inplace=True)

# rename columns
df_tot.columns = ['year', 'total']

# view the final dataframe
df_tot.head()

Với *seaborn*, việc tạo một regression plot đơn giản chỉ là gọi hàm **regplot**.

In [ ]:
sns.regplot(x='year', y='total', data=df_tot)

Đây không phải phép màu; đây là *seaborn*! Bạn cũng có thể tùy chỉnh màu của scatter plot và đường hồi quy. Hãy đổi màu sang xanh lá.

In [ ]:
sns.regplot(x='year', y='total', data=df_tot, color='green')
plt.show()

Bạn luôn có thể tùy chỉnh hình dạng marker, vì vậy thay vì marker hình tròn, hãy dùng `+`.

In [ ]:
ax = sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+')
plt.show()

Hãy phóng to biểu đồ một chút để nó đẹp mắt hơn.

In [ ]:
plt.figure(figsize=(15, 10))
sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+')
plt.show()

Và hãy tăng kích thước các marker để chúng khớp với kích thước mới của figure, đồng thời thêm tiêu đề và nhãn cho trục x và y.

In [ ]:
plt.figure(figsize=(15, 10))
ax = sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+', scatter_kws={'s': 200})

ax.set(xlabel='Year', ylabel='Total Immigration') # add x- and y-labels
ax.set_title('Total Immigration to Canada from 1980 - 2013') # add title
plt.show()

Và cuối cùng hãy tăng cỡ chữ của các nhãn tickmark, tiêu đề và nhãn trục x, y để chúng không bị bỏ rơi!

In [ ]:
plt.figure(figsize=(15, 10))

sns.set(font_scale=1.5)

ax = sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+', scatter_kws={'s': 200})
ax.set(xlabel='Year', ylabel='Total Immigration')
ax.set_title('Total Immigration to Canada from 1980 - 2013')
plt.show()

Tuyệt vời! Một scatter plot hoàn chỉnh với đường hồi quy khớp chỉ với 5 dòng code. Thật sự quá tuyệt vời phải không?

Nếu bạn không thích nền màu tím, bạn có thể dễ dàng đổi style sang nền trắng trơn.

In [ ]:
plt.figure(figsize=(15, 10))

sns.set(font_scale=1.5)
sns.set_style('ticks')  # change background to white background

ax = sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+', scatter_kws={'s': 200})
ax.set(xlabel='Year', ylabel='Total Immigration')
ax.set_title('Total Immigration to Canada from 1980 - 2013')
plt.show()

Hoặc sang nền trắng có đường lưới (gridlines).

In [ ]:
plt.figure(figsize=(15, 10))

sns.set(font_scale=1.5)
sns.set_style('whitegrid')

ax = sns.regplot(x='year', y='total', data=df_tot, color='green', marker='+', scatter_kws={'s': 200})
ax.set(xlabel='Year', ylabel='Total Immigration')
ax.set_title('Total Immigration to Canada from 1980 - 2013')
plt.show()

**Câu hỏi**: Sử dụng seaborn để tạo một scatter plot với đường hồi quy nhằm trực quan hóa tổng lượng nhập cư từ Đan Mạch (Denmark), Thụy Điển (Sweden) và Na Uy (Norway) vào Canada từ năm 1980 đến 2013.

In [ ]:
### type your answer here





<details><summary>Nhấn vào đây để xem gợi ý về các hàm cần dùng</summary>

### 💡 Gợi ý

Bạn có thể cân nhắc các hàm sau:

- <code style="color:red">loc[]</code> — chọn 3 quốc gia và các cột năm.
- <code style="color:red">transpose()</code> — hoán đổi dòng/cột.
- <code style="color:red">sum()</code> — cộng tổng theo từng dòng (`axis=1`).
- <code style="color:red">pd.DataFrame()</code> — chuyển kết quả thành dataframe.
- <code style="color:red">reset_index()</code> — dùng `inplace=True`.
- <code style="color:red">columns</code> — đổi tên cột thành `year` và `total`.
- <code style="color:red">astype()</code> — đổi cột `year` sang số nguyên.
- <code style="color:red">plt.figure()</code> — đặt kích thước khung vẽ bằng `figsize`.
- <code style="color:red">sns.set() / sns.set_style()</code> — chỉnh cỡ chữ (`font_scale`) và kiểu nền (ví dụ `whitegrid`).
- <code style="color:red">sns.regplot()</code> — vẽ scatter plot kèm đường hồi quy; dùng `x`, `y`, `data`, `color`, `marker`, `scatter_kws`.
- <code style="color:red">ax.set() / ax.set_title()</code> — đặt nhãn trục và tiêu đề.

</details>

### Cảm ơn bạn đã hoàn thành bài thực hành này!


## Tác giả

<a href="https://www.linkedin.com/in/aklson/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork970-2023-01-01">Alex Aklson</a>



## Nhật ký thay đổi


|  Ngày (YYYY-MM-DD) |  Phiên bản | Người thay đổi  |  Mô tả thay đổi |
|---|---|---|---|
| 2021-05-19  | 2.3  | Weiqing Wang  | Sửa lỗi chính tả và code smell |
| 2021-01-21  | 2.2  | Lakshmi Holla | Cập nhật ô markdown của mục lục (TOC)|
| 2020-11-03  | 2.1  | Lakshmi Holla | Thay đổi URL của tệp excel |
| 2020-08-27  | 2.0  | Lavanya  |  Chuyển bài thực hành vào repo khóa học trên GitLab |


## <h3 align="center"> © IBM Corporation 2020. Bảo lưu mọi quyền. <h3/>